# Stage 3 Qwen Model Sweep Clean (Kaggle)

Frozen-model comparison for the clean Stage 3 GT-crop protocol. The prompt, dataset, schema, and evaluator stay fixed; only `model_id` changes.

The notebook runs a one-sample preflight for each candidate first. Full validation runs only if preflight succeeds.


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
JSONL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")

PROMPT_VERSION = "qwen_vlm_labels_v1_prompt_v7f_flashover_unclear_to_unknown_nocroppath"
RUN_ID_PREFIX = "stage3_qwen_val_v2_model_sweep_clean"
MAX_PIXELS = 401408

MODEL_RUNS = [
    {"name": "qwen25vl_3b_control", "model_id": "Qwen/Qwen2.5-VL-3B-Instruct", "full_run": True},
    {"name": "qwen25vl_7b_candidate", "model_id": "Qwen/Qwen2.5-VL-7B-Instruct", "full_run": True},
]

print("Model sweep size:", len(MODEL_RUNS))


In [ ]:
def sh(args, cwd: Path | None = None, check: bool = True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run(
        [str(a) for a in args],
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def require_path(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path

def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


In [ ]:
DATA_ROOT = None
for root in DATASET_ROOT_CANDIDATES:
    if (root / JSONL_REL).exists():
        DATA_ROOT = root
        break
if DATA_ROOT is None:
    raise FileNotFoundError("Could not find clean Stage 3 JSONL in Kaggle inputs")

input_jsonl = require_path(DATA_ROOT / JSONL_REL, "clean val jsonl")
print("DATA_ROOT:", DATA_ROOT)
print("input_jsonl:", input_jsonl)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(["git", "clone", REPO_URL, str(REPO_DIR)])

sh(["python", "-m", "pip", "install", "-q", "-U", "transformers>=4.49.0", "accelerate", "qwen-vl-utils", "pillow", "pandas", "pyyaml"], cwd=REPO_DIR)
print("Repo ready:", REPO_DIR)


In [ ]:
base_cfg_path = REPO_DIR / "configs/pipeline/stage3_vlm_gt_baseline.yaml"
base_cfg = yaml.safe_load(base_cfg_path.read_text(encoding="utf-8"))

def make_model_config(model_name: str, model_id: str) -> Path:
    cfg = json.loads(json.dumps(base_cfg))
    cfg["input"]["dataset_jsonl"] = str(input_jsonl)
    cfg["prompt"]["version"] = PROMPT_VERSION
    cfg["backend"]["mode"] = "qwen_hf"
    qwen_cfg = cfg["backend"].setdefault("qwen_hf", {})
    qwen_cfg["model_id"] = model_id
    qwen_cfg["max_pixels"] = MAX_PIXELS
    qwen_cfg["torch_dtype"] = "auto"
    qwen_cfg["device_map"] = "auto"
    cfg["run"]["resume"] = False
    cfg_path = REPO_DIR / "configs" / f"stage3_model_sweep_{model_name}.yaml"
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=False), encoding="utf-8")
    return cfg_path

def metrics_row(model_name: str, model_id: str, run_id: str, preflight_ok: bool, full_ok: bool, error: str | None):
    run_dir = REPO_DIR / "outputs/stage3_vlm_baseline_runs" / run_id
    eval_dir = run_dir / "eval"
    row = {
        "model_name": model_name,
        "model_id": model_id,
        "run_id": run_id,
        "preflight_ok": preflight_ok,
        "full_ok": full_ok,
        "error": error or "",
    }
    metrics_path = eval_dir / "metrics.json"
    if metrics_path.exists():
        m = read_json(metrics_path)
        rates = m.get("rates", {})
        f1 = m.get("f1", {})
        counts = m.get("counts", {})
        row.update({
            "evaluated_total": counts.get("evaluated_total"),
            "parse_success": rates.get("parse_success_rate"),
            "schema_valid": rates.get("schema_valid_rate"),
            "coarse_acc": rates.get("coarse_class_accuracy"),
            "coarse_macro_f1": f1.get("coarse_class_macro_f1"),
            "visibility_acc": rates.get("visibility_accuracy"),
            "visibility_macro_f1": f1.get("visibility_macro_f1"),
            "needs_review_acc": rates.get("needs_review_accuracy"),
            "tag_mean_jaccard": rates.get("tag_mean_jaccard"),
            "pred_ambiguous_rate": rates.get("pred_ambiguous_rate"),
        })
    return row


In [ ]:
rows = []
for spec in MODEL_RUNS:
    model_name = spec["name"]
    model_id = spec["model_id"]
    cfg_path = make_model_config(model_name, model_id)
    preflight_id = f"{RUN_ID_PREFIX}_{model_name}_preflight"
    full_id = f"{RUN_ID_PREFIX}_{model_name}"
    preflight_ok = False
    full_ok = False
    error = None

    try:
        sh([
            "python", "scripts/run_stage3_vlm_baseline.py",
            "--config", str(cfg_path),
            "--run-id", preflight_id,
            "--max-samples", "1",
            "--no-resume",
        ], cwd=REPO_DIR)
        preflight_ok = True
    except Exception as exc:
        error = f"preflight failed: {exc}"
        print(error)

    if preflight_ok and spec.get("full_run", True):
        try:
            sh([
                "python", "scripts/run_stage3_vlm_baseline.py",
                "--config", str(cfg_path),
                "--run-id", full_id,
                "--no-resume",
            ], cwd=REPO_DIR)
            sh([
                "python", "scripts/validate_vlm_labels_v1.py",
                "--input-jsonl", str(REPO_DIR / "outputs/stage3_vlm_baseline_runs" / full_id / "predictions_vlm_labels_v1.jsonl"),
            ], cwd=REPO_DIR)
            sh([
                "python", "scripts/eval_stage3_vlm_baseline.py",
                "--gt-jsonl", str(input_jsonl),
                "--pred-jsonl", str(REPO_DIR / "outputs/stage3_vlm_baseline_runs" / full_id / "predictions_vlm_labels_v1.jsonl"),
                "--output-dir", str(REPO_DIR / "outputs/stage3_vlm_baseline_runs" / full_id / "eval"),
            ], cwd=REPO_DIR)
            full_ok = True
        except Exception as exc:
            error = f"full run failed: {exc}"
            print(error)

    rows.append(metrics_row(model_name, model_id, full_id, preflight_ok, full_ok, error))

comparison = pd.DataFrame(rows)
display(comparison)


In [ ]:
out_dir = REPO_DIR / "outputs/stage3_model_sweep_clean"
out_dir.mkdir(parents=True, exist_ok=True)
comparison_path = out_dir / "model_sweep_comparison.csv"
comparison.to_csv(comparison_path, index=False)

md_lines = ["# Stage 3 Qwen Model Sweep Clean", "", comparison.to_markdown(index=False)]
(out_dir / "model_sweep_comparison.md").write_text("\n".join(md_lines), encoding="utf-8")

completed = comparison[comparison["full_ok"] == True].copy()
if not completed.empty and "coarse_macro_f1" in completed.columns:
    ranked = completed.sort_values(["coarse_macro_f1", "coarse_acc", "tag_mean_jaccard"], ascending=[False, False, False])
    best = ranked.iloc[0].to_dict()
    print("Best completed model:", best.get("model_name"), best.get("model_id"))
    print("coarse_acc:", best.get("coarse_acc"), "coarse_macro_f1:", best.get("coarse_macro_f1"))
else:
    print("No full model run completed. Check the error column in the comparison table.")

archive_path = Path("/kaggle/working/stage3_deliverables_model_sweep_clean.tar.gz")
if archive_path.exists():
    archive_path.unlink()
sh(["tar", "-czf", str(archive_path), "-C", str(REPO_DIR / "outputs"), "stage3_model_sweep_clean", "stage3_vlm_baseline_runs"], check=True)
print("Archive:", archive_path)
